# GPU Pipeline Benchmark - Google Colab

Runs the full surveillance pipeline on a Colab GPU and records **measured** timings
(no estimates) into `Docs/gpu_benchmark_results.json` for the dissertation report.

**Stack on the `develop` branch:** YOLOv8 detection + greedy IOU tracker (ByteTrack-style,
the default backend; DeepSORT kept as the ablation baseline) + CLIP or SigLIP 2 retrieval
encoder + FAISS similarity search + rule-based anomaly engine. `configs/config.yaml` sets
`clip.device` / `yolo.device` to `auto`, so CUDA is picked up automatically on a GPU runtime.

**What gets measured (nothing is estimated):**
- Cold indexing wall time on the same workload as the CPU baseline (45-min MEVA video, 22.3 min on CPU)
- Per-stage breakdown: video decode vs YOLO detect+track vs CLIP encode
- Throughput (frames/sec) per stage and end to end
- Warm per-query latency over the 5 demo queries (20 timed runs each)
- Exact GPU model, CUDA and torch versions

**Workflow:**
1. Runtime > Change runtime type > **GPU** (note which GPU is assigned: T4 / L4 / A100)
2. Run the cells top to bottom; supply the benchmark video in step 3 (Drive upload preferred)
3. Download `Docs/gpu_benchmark_results.json` and paste it back for the report update
4. Optional: extract `cache_artifacts.zip` into the project `.cache/` on the laptop for an
   instant cache hit on the same video (cache keys are content-based, machine-independent)


In [ ]:
# 1. Verify the GPU runtime
import torch
print('CUDA available:', torch.cuda.is_available())
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('No GPU - use Runtime > Change runtime type > GPU before continuing')


In [ ]:
# 2. Clone the DEVELOP branch (latest code) and install dependencies (~3 min)
import os
if not os.path.isdir('smart_query_driven_surveillance_vlm'):
    !git clone -b develop https://github.com/jaisris/smart_query_driven_surveillance_vlm.git
%cd smart_query_driven_surveillance_vlm
!git rev-parse --abbrev-ref HEAD && git log --oneline -1

# Keep Colab's CUDA-enabled torch build: install everything except torch/torchvision
!grep -vE '^(torch|torchvision)==' requirements.txt > /tmp/reqs_colab.txt
!pip install -q -r /tmp/reqs_colab.txt


## 3. Benchmark video

**Option A (preferred, directly comparable):** upload the exact demo video
`data/videos/MEVA_bus_G340_45min.mp4` (1.77 GB) from the laptop to Google Drive.
Identical bytes means an identical workload (81,061 frames) and a valid wall-time
comparison against the 22.3 min CPU baseline.

**Option B:** re-fetch the same 9 source clips (camera G340, 2018-03-05 10:10 to 10:55)
from the public MEVA S3 bucket `mevadata-public-01` and concatenate them (next cell).
Same content, so the workload still matches.

**Option C:** any other clip (e.g. a VIRAT video). The run then reports per-stage GPU
throughput (frames/sec) only; do not derive a 45-min wall-time speedup from it.


In [ ]:
# 3a. Option A: mount Drive and point at the uploaded 45-min MEVA video
from google.colab import drive
drive.mount('/content/drive')

import os
VIDEO_PATH = '/content/drive/MyDrive/surveillance/MEVA_bus_G340_45min.mp4'  # <-- EDIT if elsewhere
if os.path.exists(VIDEO_PATH):
    print(f'Video found: {os.path.getsize(VIDEO_PATH)/1e6:.0f} MB')
else:
    print('Not found in Drive. Either run the next cell to re-fetch from the MEVA S3 bucket,')
    print('or edit VIDEO_PATH to another clip (then only throughput numbers are comparable).')


In [ ]:
# 3b. Option B (skip if the video was found above): re-fetch the 9 MEVA clips
# from the public S3 bucket and concatenate them into the same 45-min video.
import os, subprocess
if not os.path.exists(VIDEO_PATH):
    !pip install -q boto3
    import boto3
    from botocore import UNSIGNED
    from botocore.config import Config as BotoConfig

    CLIPS = [f'2018-03-05.10-{m:02d}-00.10-{m+5:02d}-00.bus.G340.r13.avi'
             for m in range(10, 55, 5)]
    BUCKET = 'mevadata-public-01'
    s3 = boto3.client('s3', config=BotoConfig(signature_version=UNSIGNED))

    # Locate the 9 keys (the drop is organised into date/hour subfolders)
    wanted, keys = set(CLIPS), {}
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=BUCKET, Prefix='drops-123-r13/2018-03-05'):
        for obj in page.get('Contents', []):
            base = obj['Key'].rsplit('/', 1)[-1]
            if base in wanted:
                keys[base] = obj['Key']
    missing = wanted - set(keys)
    assert not missing, f'Could not locate clips in bucket: {sorted(missing)}'

    os.makedirs('/content/meva_g340', exist_ok=True)
    for base in CLIPS:
        dst = f'/content/meva_g340/{base}'
        if not os.path.exists(dst):
            print('Downloading', base)
            s3.download_file(BUCKET, keys[base], dst)

    with open('/content/meva_g340/list.txt', 'w') as f:
        for base in CLIPS:
            f.write(f"file '/content/meva_g340/{base}'\n")

    VIDEO_PATH = '/content/MEVA_bus_G340_45min.mp4'
    r = subprocess.run(['ffmpeg', '-y', '-f', 'concat', '-safe', '0',
                        '-i', '/content/meva_g340/list.txt', '-c', 'copy', VIDEO_PATH])
    if r.returncode != 0:  # container copy failed -> re-encode (slower, same frame count)
        subprocess.run(['ffmpeg', '-y', '-f', 'concat', '-safe', '0',
                        '-i', '/content/meva_g340/list.txt',
                        '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '20', VIDEO_PATH],
                       check=True)
    print('Ready:', VIDEO_PATH, f'({os.path.getsize(VIDEO_PATH)/1e6:.0f} MB)')
else:
    print('Video already available - skipping S3 fetch')


## 4. Benchmark run

Cold indexing (fresh cache directory, so nothing is served from cache) with per-stage
timing, then warm per-query latency. The config is **identical** to the CPU baseline
(`frame_skip` 15, adaptive cap 4000, CLIP batch size 32, ByteTrack backend); only the
device differs. Timing wrappers are installed from the notebook - no repo code changes.
Model weights are pre-downloaded so the timed run measures compute, not network transfer
(the CPU baseline also loaded weights from local disk).


In [ ]:
# 4. GPU benchmark: writes Docs/gpu_benchmark_results.json
import os, sys, time, json, shutil, hashlib, statistics
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
sys.path.insert(0, '.')
import numpy as np
import torch

assert torch.cuda.is_available(), 'GPU runtime required'
GPU_NAME = torch.cuda.get_device_name(0)

from utils.config_loader import get_config
config = get_config()

# Fresh cache dir -> guaranteed cold run
BENCH_CACHE = '/content/bench_cache'
shutil.rmtree(BENCH_CACHE, ignore_errors=True)
config.pipeline.cache_dir = BENCH_CACHE
print(f'GPU: {GPU_NAME} | tracker: {config.tracking.backend} | encoder: {config.clip.model_name}')

# Pre-download model weights (excluded from the timed run)
from models.clip_encoder import CLIPEncoder
from ultralytics import YOLO
_warm = CLIPEncoder(config); del _warm
YOLO(config.yolo.model)
torch.cuda.empty_cache()

# ---- Per-stage timers: wrap decode / detect+track / CLIP encode ----
from data.video_loader import VideoLoader
from pipeline.frame_processor import FrameProcessor

STAGE = {'decode_sec': 0.0, 'detect_track_sec': 0.0, 'clip_encode_sec': 0.0}
COUNT = {'frames_decoded': 0, 'frames_clip_encoded': 0}

if not hasattr(VideoLoader, '_bench_orig_iter'):
    VideoLoader._bench_orig_iter = VideoLoader.iter_frames
    def _timed_iter(self, *a, **kw):
        it = VideoLoader._bench_orig_iter(self, *a, **kw)
        while True:
            t0 = time.perf_counter()
            try:
                item = next(it)
            except StopIteration:
                return
            STAGE['decode_sec'] += time.perf_counter() - t0
            COUNT['frames_decoded'] += 1
            yield item
    VideoLoader.iter_frames = _timed_iter

if not hasattr(FrameProcessor, '_bench_orig_process'):
    FrameProcessor._bench_orig_process = FrameProcessor.process
    def _timed_process(self, *a, **kw):
        t0 = time.perf_counter()
        out = FrameProcessor._bench_orig_process(self, *a, **kw)
        STAGE['detect_track_sec'] += time.perf_counter() - t0
        return out
    FrameProcessor.process = _timed_process

if not hasattr(CLIPEncoder, '_bench_orig_batch'):
    CLIPEncoder._bench_orig_batch = CLIPEncoder.encode_image_batch
    def _timed_batch(self, frames_rgb):
        t0 = time.perf_counter()
        out = CLIPEncoder._bench_orig_batch(self, frames_rgb)
        STAGE['clip_encode_sec'] += time.perf_counter() - t0
        COUNT['frames_clip_encoded'] += len(frames_rgb)
        return out
    CLIPEncoder.encode_image_batch = _timed_batch

for k in STAGE: STAGE[k] = 0.0
for k in COUNT: COUNT[k] = 0

from pipeline.video_pipeline import VideoPipeline

def hash_file(path, chunk_mb=8):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(chunk_mb * 1024 * 1024):
            h.update(chunk)
    return h.hexdigest()

print('Hashing video (not counted in the indexing wall time, same as the CPU baseline) ...')
content_hash = hash_file(VIDEO_PATH)

pipe = VideoPipeline(config)
t0 = time.perf_counter()
result = pipe.run(VIDEO_PATH, content_hash=content_hash)
wall_sec = time.perf_counter() - t0

meta = result.video_metadata
clip_device = pipe._get_encoder().device
other_sec = wall_sec - sum(STAGE.values())

print(f'\n=== COLD INDEXING RUN (device={clip_device}) ===')
print(f'wall: {wall_sec:.1f} s ({wall_sec/60:.2f} min)')
for name, sec in STAGE.items():
    print(f'  {name:>16}: {sec:7.1f} s ({100*sec/wall_sec:4.1f}%)')
print(f'  {"other_sec":>16}: {other_sec:7.1f} s (model load, static-frame check, anomaly, I/O)')
print(f'frames decoded/processed: {COUNT["frames_decoded"]}, CLIP-encoded: {COUNT["frames_clip_encoded"]}')
print(f'tracks: {len(result.track_histories)}, anomalies: {len(result.anomaly_events)}')

throughput = {
    'decode_fps': round(COUNT['frames_decoded'] / STAGE['decode_sec'], 1) if STAGE['decode_sec'] else None,
    'detect_track_fps': round(COUNT['frames_decoded'] / STAGE['detect_track_sec'], 1) if STAGE['detect_track_sec'] else None,
    'clip_encode_fps': round(COUNT['frames_clip_encoded'] / STAGE['clip_encode_sec'], 1) if STAGE['clip_encode_sec'] else None,
    'sampled_frames_per_sec_end_to_end': round(COUNT['frames_decoded'] / wall_sec, 1),
    'video_sec_indexed_per_wall_sec': round(meta.duration_sec / wall_sec, 1),
}

# ---- Warm per-query latency over the 5 demo queries ----
from retrieval.query_encoder import QueryEncoder
from retrieval.similarity_search import SimilaritySearch
from retrieval.temporal_localizer import localize_segments

search = SimilaritySearch(config)
search.build_index(result.embedding_matrix, result.frame_index_entries)
qenc = QueryEncoder(encoder=pipe._get_encoder(), config=config)

DEMO_QUERIES = ['people waiting at a bus stop', 'a bus arriving', 'a person walking alone',
                'a group of people talking', 'a car driving past']

for q in DEMO_QUERIES:  # warm-up pass (CUDA kernels, tokenizer)
    localize_segments(search.search(qenc.encode(q), top_k=20), min_score=0.20)

query_latency, queries_top3, all_ms = {}, [], []
for q in DEMO_QUERIES:
    times_ms = []
    for _ in range(20):
        t = time.perf_counter()
        segs = localize_segments(search.search(qenc.encode(q), top_k=20), min_score=0.20)
        times_ms.append((time.perf_counter() - t) * 1000)
    all_ms += times_ms
    query_latency[q] = {'mean_ms': round(statistics.mean(times_ms), 1),
                        'median_ms': round(statistics.median(times_ms), 1),
                        'min_ms': round(min(times_ms), 1),
                        'max_ms': round(max(times_ms), 1)}
    queries_top3.append({'query': q,
                         'top3': [{'start': round(s.start_sec, 1), 'end': round(s.end_sec, 1),
                                   'score': round(s.peak_score, 4)} for s in segs[:3]]})

print('\n=== WARM QUERY LATENCY (encode + FAISS search + localize, 20 runs/query) ===')
for q, st in query_latency.items():
    print(f'  {st["median_ms"]:6.1f} ms median ({st["min_ms"]:.1f}-{st["max_ms"]:.1f})  "{q}"')

# ---- CPU baselines already in the dissertation report (measured locally) ----
CPU_BASELINES = {
    'meva_45min_indexing_wall_min': 22.3,
    'traffic_clip_58s_cold_sec': 132,
    'virat_70s_deepsort_sec': 98.2,
    'virat_70s_bytetrack_sec': 67.7,
    'query_latency_ms_range': [90, 140],
}

is_meva_45 = 'MEVA' in os.path.basename(VIDEO_PATH) and 40 <= meta.duration_sec / 60 <= 50
comparison = None
if is_meva_45:
    comparison = {
        'workload': '45-min MEVA video, identical config (frame_skip, tracker, encoder)',
        'cpu_wall_min': CPU_BASELINES['meva_45min_indexing_wall_min'],
        'gpu_wall_min': round(wall_sec / 60, 2),
        'indexing_speedup_x': round(CPU_BASELINES['meva_45min_indexing_wall_min'] * 60 / wall_sec, 1),
        'cpu_query_latency_ms_range': CPU_BASELINES['query_latency_ms_range'],
        'gpu_query_latency_median_ms': round(statistics.median(all_ms), 1),
    }
    print('\n=== CPU vs GPU (45-min MEVA, same workload) ===')
    print(f'  Indexing wall : {comparison["cpu_wall_min"]} min (CPU)  ->  {comparison["gpu_wall_min"]} min ({GPU_NAME})')
    print(f'  Speedup       : {comparison["indexing_speedup_x"]}x')
    print(f'  Query latency : {CPU_BASELINES["query_latency_ms_range"]} ms (CPU)  ->  {comparison["gpu_query_latency_median_ms"]} ms median (GPU)')
else:
    print('\nNOTE: workload is not the 45-min MEVA video. Report the throughput numbers only;'
          ' do not derive a wall-time speedup vs the CPU baseline from this run.')

out = {
    'gpu': GPU_NAME,
    'cuda_version': torch.version.cuda,
    'torch_version': torch.__version__,
    'device_used': clip_device,
    'video': os.path.basename(VIDEO_PATH),
    'duration_min': round(meta.duration_sec / 60, 1),
    'frames_total': meta.total_frames,
    'frames_decoded': COUNT['frames_decoded'],
    'frames_indexed': int(result.embedding_matrix.shape[0]),
    'unique_tracks': len(result.track_histories),
    'anomaly_events': len(result.anomaly_events),
    'config': {'tracker_backend': config.tracking.backend,
               'clip_model': config.clip.model_name,
               'frame_skip': config.pipeline.frame_skip,
               'max_indexed_frames': config.pipeline.max_indexed_frames,
               'batch_size': config.pipeline.batch_size,
               'yolo_model': config.yolo.model},
    'indexing': {'wall_sec': round(wall_sec, 1),
                 'wall_min': round(wall_sec / 60, 2),
                 'stage_sec': {k: round(v, 1) for k, v in STAGE.items()} | {'other_sec': round(other_sec, 1)},
                 'throughput': throughput},
    'query_latency_ms': query_latency,
    'query_latency_overall_median_ms': round(statistics.median(all_ms), 1),
    'queries': queries_top3,
    'cpu_baselines_from_report': CPU_BASELINES,
    'cpu_vs_gpu': comparison,
}
os.makedirs('Docs', exist_ok=True)
with open('Docs/gpu_benchmark_results.json', 'w') as f:
    json.dump(out, f, indent=2)
print('\nSAVED -> Docs/gpu_benchmark_results.json  (download this file / paste it back)')


In [ ]:
# 5. (Optional) UCF-Crime evaluation on GPU - full dataset is feasible here.
# Upload/extract the Kaggle dataset (odins0n/ucf-crime-dataset) to Drive first,
# then point UCF_TEST at its Test folder.
import os
UCF_TEST = '/content/drive/MyDrive/surveillance/archive/Test'  # <-- EDIT IF USING
if os.path.isdir(UCF_TEST):
    !python evaluation/run_ucf_eval.py --data-root "$UCF_TEST" --frames-per-class 2000 --batch-size 256
else:
    print('UCF dataset not found in Drive - skipping (edit UCF_TEST above to enable)')


In [ ]:
# 6. Bundle the results for download. If you used Option A (identical bytes), the cache
# is portable: extract the zip's .cache/ into the project's .cache/ on the laptop for an
# instant cache hit in the local Streamlit UI.
!mkdir -p .cache && cp -r /content/bench_cache/. .cache/ 2>/dev/null
!zip -r cache_artifacts.zip .cache Docs/gpu_benchmark_results.json Docs/ucf_eval_results.json Docs/ucf_roc_curve.png 2>/dev/null
from google.colab import files
files.download('cache_artifacts.zip')
